# ns-3 5G NR Interactive Visualization Dashboard

**GSoC 2026 Proof-of-Concept** | ns-3 Project — "Enabling 5G NR Examples Visualization"

This notebook parses real ns-3 NR simulation traces and provides interactive visualizations.
Trace format:  +  from [cttc-lena/nr](https://gitlab.com/cttc-lena/nr).

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from ns3_nr_parser import (
    load_rlc_stats, load_sinr_stats, load_topology,
    load_flow_monitor, compute_cell_stats, compute_jains_fairness, print_summary
)

data_dir = Path("data")
dl_rlc  = load_rlc_stats(data_dir / "DlRlcStats.txt")
ul_rlc  = load_rlc_stats(data_dir / "UlRlcStats.txt")
dl_sinr = load_sinr_stats(data_dir / "DlPhySinr.txt")
topo    = load_topology(data_dir / "topology.json")
flows   = load_flow_monitor(data_dir / "flow_monitor.json")

print("✓ All trace files loaded")
print_summary(dl_rlc, ul_rlc, dl_sinr)

## 1. Simulation Summary Table

In [ ]:
cell_stats = compute_cell_stats(dl_rlc)
summary = dl_rlc.groupby("IMSI").agg(
    serving_cell=("cellId", lambda x: x.mode()[0]),
    mean_dl_tput=("DL_throughput_mbps", "mean"),
    mean_delay_ms=("delay_ms", "mean"),
    mean_loss_pct=("packet_loss_pct", "mean"),
).reset_index()
ul_summary = ul_rlc.groupby("IMSI").agg(mean_ul_tput=("DL_throughput_mbps", "mean")).reset_index()
summary = summary.merge(ul_summary, on="IMSI")
summary.columns = ["IMSI","Cell","DL Tput (Mbps)","Delay (ms)","Loss (%)","UL Tput (Mbps)"]
summary = summary.round(2)
print(summary.to_string(index=False))

## 2. Per-UE DL Throughput Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
cmap = matplotlib.colormaps.get_cmap("tab10")
for idx, imsi in enumerate(sorted(dl_rlc["IMSI"].unique())):
    d = dl_rlc[dl_rlc["IMSI"]==imsi]
    ax.plot(d["start_time"], d["DL_throughput_mbps"], label=f"UE {imsi}",
            alpha=0.85, linewidth=1.2, color=cmap(idx/10))
ax.axvline(x=2.5, color="black", linestyle="--", linewidth=2, label="Handover (UE 3)")
ax.set_xlabel("Time [s]"); ax.set_ylabel("Throughput [Mbps]")
ax.set_title("Per-UE DL Throughput", fontweight="bold")
ax.legend(ncol=3, fontsize=8); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("figures/notebook_tput.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figure saved")

## 3. SINR Distribution (Box Plot per Cell)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
cell_sinr_data = [dl_sinr[dl_sinr["cellId"]==c]["sinr_dB"].values for c in [1,2,3]]
bp = ax.boxplot(cell_sinr_data, labels=["Cell 1","Cell 2","Cell 3"],
                patch_artist=True, notch=True)
colors = ["#E63946","#457B9D","#2A9D8F"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_ylabel("SINR [dB]"); ax.set_title("SINR Distribution per Cell", fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig("figures/notebook_sinr_box.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Jain's Fairness Index Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
jfi = compute_jains_fairness(dl_rlc)
ax.plot(jfi.index, jfi.values, color="#6A4C93", linewidth=2.5)
ax.fill_between(jfi.index, jfi.values, alpha=0.2, color="#6A4C93")
ax.axhline(y=1.0, color="green", linestyle="--", label="Perfect fairness (JFI=1.0)")
ax.axvline(x=2.5, color="black", linestyle=":", linewidth=2, label="Handover event")
ax.set_ylim(0, 1.05)
ax.set_xlabel("Time [s]"); ax.set_ylabel("JFI")
ax.set_title("Jain's Fairness Index", fontweight="bold")
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("figures/notebook_fairness.png", dpi=120, bbox_inches="tight")
plt.show()

## 5. Full Dashboard

In [ ]:
from visualize_nr import build_dashboard
build_dashboard()
print("
✓ Full dashboard generated: figures/nr_dashboard.png")

## 6. Animated Handover Visualization

In [ ]:
from animate_handover import build_animation
build_animation()
print("
✓ Animation generated: figures/handover_animation.gif")